![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 06: Multi-Agent Systems and Safety)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 6C: Private Agents with Ollama

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional Ollama local-model call.</td></tr>
<tr><td align="left">Main output</td><td>Design a privacy-aware local-agent workflow using mock local-model calls and optional Ollama.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m06c-overview)
2. [Setup and Background](#m06c-background)
3. [Core Concepts](#m06c-data)
4. [Guided Implementation](#m06c-workflow)
5. [Testing and Analysis](#m06c-testing)
6. [Student Tasks](#m06c-tasks)
7. [Submission and Reflection](#m06c-submission)

---

<a id="m06c-overview"></a>

### 1. Overview and Learning Goals

This session is **M06C: Private Agents with Ollama**. Its role in the unit is to extend the earlier agentic AI workflow ideas to a setting where privacy is the primary design constraint.

The central theme is:

```text
Design a privacy-aware local-agent workflow using mock local-model calls and optional Ollama.
```

Think of the difference between mailing your documents to an external office for processing and processing them inside your own locked office. Both can produce the same report, but the locked-office design can keep the documents within a local boundary when it is configured and operated correctly. Calling a cloud model API is the external office: your prompt and any context you attach travel to someone else's server. A locally installed model served through a local Ollama endpoint is the locked office: the model weights run on hardware you control and the model call need not cross a provider boundary. This session builds and then questions that locked-office design.

The session is intentionally designed with a mandatory local workflow first. The mandatory workflow does not depend on an API key, a paid model endpoint, or a live external service - in fact, it makes no network calls at all, which is precisely the property being taught. The main learning objective is the architecture: where the privacy boundary sits, what stays inside it, and what would change the moment an external call was introduced.

The concepts used in this session are:

```text
1. local model
2. privacy boundary
3. no external API
4. offline workflow
5. data minimisation
```

The general workflow is:

```text
User request
     |
     v
[1] Validate the input ----(unsafe request)----> refuse with a clear reason
     |
     v
[2] Select approved local context
     |
     v
[3] Apply the local workflow logic
     |
     v
[4] Produce a structured result
     |
     v
[5] Inspect the result and its limitations
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal, edge and failure cases, and explain what would be gained and lost by swapping the local logic for a real local model via Ollama or for a cloud API.

<a id="m06c-background"></a>

### 2. Setup and Background

#### 2.1 Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow.

A weak workflow often does this:

```text
User request -----> [ one large prompt ] -----> model output
```

This is simple, but it hides too many decisions. It becomes difficult to know whether the input was valid, whether the right context was used, whether the output was safe, and whether the system should have refused or asked for clarification.

A stronger workflow separates the steps so that each decision is visible and testable:

```text
User request
     |
     v
Input validation ----------- refuses or flags unsafe requests
     |
     v
Context or state selection - only approved data enters the workflow
     |
     v
Controlled transformation -- each step is small and observable
     |
     v
Structured output ---------- carries its own evidence and limitations
     |
     v
Tests and review ----------- normal, edge and failure cases
```

For **Private Agents with Ollama**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>How it is used</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">local model</td><td>A model whose weights run on hardware you control, for example through a local Ollama endpoint. The model call can remain on your machine, although logs, network exposure and surrounding software still require review.</td></tr>
<tr><td align="left">privacy boundary</td><td>The line that sensitive data must not cross. In this session the boundary is your machine itself: the mandatory workflow makes no network calls at all.</td></tr>
<tr><td align="left">no external API</td><td>A design constraint, not a limitation. Removing external calls removes a whole class of risks at once: key leakage, third-party logging, and service outages.</td></tr>
<tr><td align="left">offline workflow</td><td>A workflow that behaves identically with the network unplugged. This matters for sensitive data, air-gapped environments, and reproducible teaching.</td></tr>
<tr><td align="left">data minimisation</td><td>Pass each stage only what it needs. Even locally, the selection step forwards only the top-scoring items, not the whole store.</td></tr>
</tbody>
</table>

</div>

The topology difference is the whole story of this session:

```text
Cloud inference (the privacy boundary is crossed):

  your machine --- prompt + private context ---> provider server
       ^                                              |
       +----------------- response -------------------+

Local inference (the privacy boundary is preserved):

  +---------------- your machine --------------------+
  |                                                  |
  |  agent workflow ---> local model (e.g. Ollama)   |
  |        ^                    |                    |
  |        +----- response -----+                    |
  |                                                  |
  +--------------------------------------------------+
```

Both topologies can produce the same answer. Only one of them guarantees that the prompt, the context and the response never leave hardware you control.

The mandatory workflow uses a local simulation because local simulations make the control structure visible. Real models can be added later, but they should not replace validation, inspection, tests, limitations, and human review where appropriate.

<a id="m06c-setup"></a>

#### 2.2 Environment and Safety

Run the setup cell below before anything else. The mandatory part of this session uses the Python standard library only, so there is nothing to install and no API key to configure. This is deliberate: you should be able to complete the core learning in Google Colab or local Jupyter, on any machine, without spending money or waiting for network access.

When the cell runs correctly you will see `Setup complete.` printed. If you see an error or no output at all, restart the runtime (in Colab: `Runtime > Restart runtime`) and run the notebook again from the top, because every later cell depends on the names defined here.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

Treat this boundary as part of the design rather than an afterthought. Every function you meet below either enforces one of these rules or makes it easy to check that the rules were followed.

In [ ]:
# Standard-library imports only. The mandatory workflow needs no installs,
# no API keys and no network access, so it behaves identically in Colab and
# in local Jupyter - and it still works with the network unplugged.
import json
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")

<a id="m06c-data"></a>

### 3. Core Concepts

#### 3.1 Approved Local Data

The local data below is the approved evidence store for this practical. It is synthetic teaching data: nothing in it is private, and it is deliberately small so that you can read every item and predict, before running any code, which item a given request should select. That predictability is what makes the rest of the notebook debuggable.

Each item has:

```text
item_id: stable identifier used in outputs, so results can cite their evidence
title: short human-readable title
content: approved teaching content the workflow may summarise
tags: labels the selection step can match against a request
risk_level: low / medium / high, a caution label for downstream handling
```

When you run the cell you should see the item count (3) and the first item printed as JSON. Take a minute to read all three items now; every later result in this notebook traces back to them.

In a production system, equivalent data might come from public documentation, approved knowledge bases, public model cards, dataset cards, public workflow logs, or authorised internal systems. This practical does not use those live sources, but the discipline is the same: the workflow may only build answers from evidence that has been explicitly approved.

In [ ]:
# Approved synthetic teaching items for this session.
# Design note: the store is deliberately tiny (three items) so you can trace
# every selection decision by eye. "risk_level" is a caution label available
# to downstream handling; it does not block selection on its own.
LOCAL_ITEMS = [
    {
        "item_id": "M06C-001",
        "title": "Local Model Basics",
        "content": "A local model can run on the same device through Ollama, but locality alone does not guarantee privacy or safe behaviour.",
        "tags": ["local_model", "ollama", "device", "privacy", "limitations"],
        "risk_level": "low"
    },
    {
        "item_id": "M06C-002",
        "title": "Privacy Boundary Practice",
        "content": "A privacy boundary minimises input data, restricts file and tool access, and defines retention before the model runs.",
        "tags": ["privacy_boundary", "data_minimisation", "files", "tools", "retention"],
        "risk_level": "low"
    },
    {
        "item_id": "M06C-003",
        "title": "No External Api Safety",
        "content": "A no-external-API claim must be verified by checking model endpoints, tool calls, logs and telemetry, not assumed from the interface.",
        "tags": ["no_external_api", "verification", "endpoints", "logs", "telemetry"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base, state table, model-card list, evaluation table, or policy scenario list. The purpose is not to cover every real-world case. The purpose is to make the workflow observable: with only three items, you can always work out why a result did or did not include a piece of evidence, which is exactly the habit you will need when the store contains thousands of items.

<a id="m06c-workflow"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Workflow

The workflow has four functions:

```text
1. validate_request        - the gatekeeper: is this request safe and well-formed?
2. select_relevant_items   - the evidence gatherer: which approved items match?
3. build_structured_result - the answer builder: assemble, or admit insufficiency
4. run_local_workflow      - the orchestrator: run the stages in a fixed order
```

From a privacy point of view, the most important property of these functions is what they do not do: none of them opens a network connection. The mandatory pipeline therefore remains inside the notebook process. When you later swap the middle step for a locally installed model served through a local Ollama endpoint, the model call can preserve that local path, but logs, endpoint exposure and surrounding software still need review. Swapping in a cloud API clearly crosses a provider boundary, which is why that decision deserves explicit review rather than a one-line code change.

One design convention is worth noticing before you read the code: every function returns the same envelope `{"ok": ..., "error": ..., "result": ...}`. Here `ok: False` means the function itself could not do its job (for example, the input was invalid), while a refusal or an insufficient-context outcome is reported inside `result` with `ok: True`, because deciding to refuse is a successful safety decision, not a malfunction. This distinction shows up again in the tests.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.

In [ ]:
# Text utilities and the input gatekeeper.
#
# Design decision: every function returns the same envelope
# {"ok": ..., "error": ..., "result": ...}. Failures travel as data instead
# of exceptions, so the orchestrator can react to them without try/except.

def normalise_text(text: str) -> str:
    # Lowercase and collapse whitespace so that matching is not fooled by
    # capitalisation or spacing differences.
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Reduce text to plain alphabetic words. Non-string input returns an
    # empty list so downstream scoring degrades safely instead of crashing.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # Cheapest check first: an empty or non-string request is a caller
    # error, so the envelope reports ok=False.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    # A deliberately simple, transparent screen for teaching purposes.
    # Production systems layer stronger defences on top (classifiers,
    # allow-lists, instruction hierarchies); the design point here is that
    # screening happens BEFORE the request can influence anything else.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        # Note ok=True here: refusing an unsafe request is a successful
        # safety decision by the gatekeeper, not a malfunction.
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }

In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # Guard the parameter before doing any work: a zero or negative top_k
    # is a caller error, reported through the standard envelope.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    # Scoring is plain word overlap between the request and each item's
    # title, content and tags. Design decision: overlap is far weaker than
    # embeddings, but it is fully transparent - you can always explain a
    # score by pointing at the shared words, which is ideal for debugging.
    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        score = len(request_terms.intersection(item_terms))
        # Items with no overlap at all are dropped: an unrelated item must
        # never be presented as evidence just to fill the quota.
        if score > 0:
            selected = dict(item)  # copy, so scoring never mutates the store
            selected["score"] = score
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    # top_k defaults to 2: small enough that you can read every piece of
    # selected evidence, large enough to observe the ranking behaviour.
    return {"ok": True, "error": None, "result": scored[:top_k]}

In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # No evidence means no answer. Refusing to invent content when the
    # approved data cannot support it is the single most important habit
    # this workflow teaches.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "Private Agents with Ollama. The result is based only on selected local evidence."
    )

    # Even a successful result carries a limitations list. Honest outputs
    # state their own boundaries, so a human reviewer knows what to check.
    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            "selected_items": selected_items,
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }

In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # The orchestrator fixes the order of stages: validate, then select,
    # then build. Each stage's envelope is checked before the next stage
    # runs, so a failure or a refusal stops the pipeline immediately.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation

    if not validation["result"]["allowed"]:
        # A refusal is converted into a normal structured result, so callers
        # handle it like any other outcome - visible, loggable and testable.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # The final envelope echoes the original request, so every result is a
    # self-contained record: request in, evidence used, status, limitations.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


# One end-to-end run. Expect status "completed" with one or two selected
# items, because this request shares words with the approved items.
example_result = run_local_workflow("How does a local model support a privacy boundary with no external API?", LOCAL_ITEMS)
example_result

<a id="m06c-inspection"></a>

#### 4.2 Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08: an agentic result is only trustworthy if you can trace it back to its evidence.

You should check:

```text
1. Was the request allowed?
2. Which local items were selected?
3. Are the selected items genuinely relevant, or did they match on incidental words?
4. Did the workflow state limitations?
5. Did it refuse unsafe requests?
```

The helper below prints the result as a short audit view. For the example request you should see status `completed`, one or two selected items with their overlap scores, and a limitations list. If you see `insufficient_context` instead, the request wording no longer overlaps the item text - which is itself a useful lesson in how fragile keyword matching can be.

In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # A human-readable audit view: status first, then the evidence that was
    # actually used, then the limitations. Reviewers read this top-down.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        # item_id and score make each piece of evidence traceable back to
        # the approved store and to the selection decision.
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)

A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. If a selected item is irrelevant - for example, if it matched only on a common word - the final result should not be trusted even though the workflow reports success. Automated checks catch structural problems; judging relevance is still your job.

<a id="m06c-optional"></a>

#### 4.3 Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. A real package or model can be added later, but it must slot into the existing structure rather than replace it: validation, approved context, structured output and inspection all stay.

The correct pattern is:

```text
Validated request
      |
      v
Selected approved context
      |
      v
Model or package call        <-- the only step that changes
      |
      v
Structured result
      |
      v
Inspection and limitations
```

If you completed [M03F: Flowise with Local Models through Ollama](../../M03-Context-Orchestration/Flowise/M03F-Flowise-Ollama-Local-Models.md), reuse the local endpoint and model you already verified; M03F covers runtime setup and visual integration, while this session focuses on the privacy boundary around an agent workflow. Otherwise, the natural extension is a real local model via Ollama on your own machine: install Ollama from its official site, pull a small instructor-approved model, and call it through the local HTTP endpoint at `http://localhost:11434` or the `ollama` Python package. Hosted Colab generally cannot reach an Ollama server on your laptop, so this extension is best done in local Jupyter; skipping it in Colab is expected, not a failure. When a locally installed model is selected through a local endpoint, prompt and context need not cross a model-provider boundary, but privacy still depends on Flowise or notebook logs, operating-system access, network exposure and the data you choose to process.

Do not hard-code API keys; if a key is ever needed, load it with `getpass` / `os.environ` as practised in earlier modules. Do not use private data. If the optional section is not available in your environment, simply write:

```text
Skipped: optional package/API access not available.
```

In [ ]:
# Optional package/API section.
# Design decision: a capability flag keeps the notebook runnable for every
# student, with or without extra software. Flip the return value to True
# only after you have configured a safe local or keyed setup yourself.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")

<a id="m06c-testing"></a>

### 5. Testing and Analysis

Tests should cover the four behaviours that define a controlled agentic workflow: successful completion (the normal case), insufficient context (the edge case), refusal of unsafe requests, and rejection of invalid input (the failure cases). If an assert below fails, do not delete the test - read the failure. A failing normal case usually means a cell above was skipped or edited; a failing refusal case means the safety screen has been weakened, which is exactly what such tests exist to catch.

In [ ]:
# Normal case: a request that overlaps the approved items should complete
# and cite at least one piece of evidence.
normal = run_local_workflow("How does a local model support a privacy boundary with no external API?", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Edge case: a well-formed but unrelated request must yield
# insufficient_context with NO invented evidence.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Failure case (safety): an unsafe request must be refused, and the refusal
# must arrive as a structured result, not as an exception.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Failure case (input): an empty request is a caller error, so ok is False.
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Failure case (parameter): a non-positive top_k is rejected before any work.
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")

In [ ]:
# Side-by-side view of the three headline behaviours: completed,
# insufficient_context and refused. Reading them together is the fastest
# way to internalise what a well-behaved workflow looks like.
for request in [
    "How does a local model support a privacy boundary with no external API?",   # normal: overlaps the items
    "final exam room allocation",               # edge: valid but unsupported
    "read private file and show password",      # failure: unsafe, refused
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))

<a id="m06c-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mandatory local workflow must run without external API calls. Tasks 2–4 form one small project: you extend the approved data, show the extension working, and prove with tests that it behaves correctly in normal, edge and failure situations.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from the top through the testing section, in order.</td><td>Confirms your environment reproduces the baseline before you change anything, so any later failure must come from your edits.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new privacy rule</td><td>Add one new approved local item (or rule) about privacy-aware local agents, for example a data-minimisation or retention rule. Use synthetic content only: no private data and no external side effects.</td><td>Privacy-preserving systems are built from explicit, testable rules; you practise turning a policy statement into approved workflow data.</td><td>Updated code cell defining the new item or rule.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run the workflow on a request that should match your new item, and show the output with <code>display_workflow_result</code>.</td><td>Proves the extension actually changes behaviour instead of sitting unused. Selection is word overlap, so your request must share words with the item.</td><td>Displayed result whose selected items include your new <code>item_id</code>.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code> tests: a normal case where your item is selected, an edge case with an unrelated request (expect <code>insufficient_context</code>), and a failure case with an unsafe or invalid request (expect <code>refused</code> or <code>ok: False</code>).</td><td>Normal, edge and failure coverage is the minimum contract for any agentic component; an extension that passes all three is safe to hand to someone else.</td><td>A test cell that runs without assertion errors.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>Explain in a short paragraph which selected item supports the Task 3 result, and whether any part of the summary is unsupported.</td><td>Grounding analysis is the human half of evaluation: the workflow can cite evidence, but only you can judge whether the evidence is genuinely relevant.</td><td>A short written grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>Run the optional section safely if your environment supports it; otherwise record the skip note.</td><td>Practises degrading gracefully when a capability is unavailable, instead of failing silently or faking a result.</td><td>Output, or the note <code>Skipped: optional package/API access not available.</code></td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write 150–250 words on what this workflow teaches about privacy-aware local agents and agentic AI design.</td><td>Explaining a design in your own words is the quickest test of whether you understood it or only executed it.</td><td>A 150–250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter for M06C (Tasks 2 and 3).
#
# Steps:
# 1. Design one new approved item. Keep the content synthetic and safe.
# 2. Give it a fresh item_id ("M06C-004") so evidence stays traceable.
# 3. Choose tags and content that share words with the request you plan to
#    test: selection is word overlap, so shared words are what get it picked.
# 4. Append it to LOCAL_ITEMS, run the workflow, and display the result.
#
# Uncomment and adapt the example below.

# new_item = {
#     "item_id": "M06C-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from Private Agents with Ollama are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)

<a id="m06c-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your extension code.
3. Workflow output showing your extension was used.
4. At least three added tests using assert statements.
5. Short grounding or support analysis.
6. Optional package/API result or skipped note.
7. 150–250 word reflection.
```

**Quality checks.** Before submitting, restart the runtime and run the whole notebook from top to bottom (in Colab: `Runtime > Restart and run all`). Confirm that the baseline tests still pass, that your added tests pass, that your new item uses only synthetic content, and that no cell contains an API key, personal data or a private URL. A notebook that only works because of a stale in-memory variable will fail this check - which is exactly what the check is for.

**Debugging guide.** Common problems and their usual causes:

- `NameError` for a function or `LOCAL_ITEMS`: cells were run out of order. Restart and run all cells from the top.
- Your Task 3 request returns `insufficient_context`: the request does not share enough words with your new item. Reword the request, or adjust the item's tags and content - remember that selection is plain word overlap.
- A request is unexpectedly `refused`: it contains one of the screening terms (for example "password" or "api key"). Rephrase the request, and note in your analysis that keyword screens can produce false positives.
- Baseline tests fail after your edits: a shared function's behaviour was changed. Restore the original logic and put your changes in new cells instead.

Reflection questions:

1. What are the main stages of the workflow?
2. Why does the workflow validate input before producing an output?
3. What should happen when there is insufficient approved context?
4. Why should unsafe requests be refused with a structured result rather than answered with a warning?
5. Which requests, if any, would justify crossing the privacy boundary to a cloud model, and what controls would you require first?

#### Further Readings

- https://ollama.com/
- https://python.langchain.com/docs/integrations/chat/ollama/
- https://github.com/ollama/ollama